# Sifter Redrob Learned Reranker Training

Use this notebook in Google Colab with a GPU runtime. It prepares Redrob training data, fine-tunes a Hugging Face reward/reranker model, and pushes it to the Hub.

In [ ]:
%cd /content
![ -d Sifter_Redrob_Hackathon ] || git clone https://github.com/shikhar1809/Sifter_Redrob_Hackathon.git
%cd /content/Sifter_Redrob_Hackathon
!git pull
!pip install -q "transformers>=4.41" "datasets>=2.19" "accelerate>=0.30" "sentencepiece" "protobuf" "scikit-learn>=1.5,<1.9" "scipy" "huggingface_hub"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# This prints candidate files from Drive. Copy the correct path into CANDIDATES below if needed.
!find /content/drive/MyDrive -name "candidates.jsonl" | head -20

In [ ]:
# Fast Colab default: 2,000 candidates. Increase later after the demo model works.
import os

CANDIDATES = '/content/drive/MyDrive/redrob/candidates.jsonl'
assert os.path.exists(CANDIDATES), f'Candidate file not found: {CANDIDATES}. Use the find cell above and paste the real path here.'

!python ml/prepare_redrob_preference_data.py \
  --candidates "$CANDIDATES" \
  --candidate-pages-dir apps/web/public/redrob-candidate-pages \
  --labels-csv ml/recruiter_labels_template.csv \
  --out-dir data/redrob-reranker \
  --max-records 2000 \
  --seed 42

!ls data/redrob-reranker

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_USERNAME = 'shikharshahi'
MODEL_ID = f'{HF_USERNAME}/sifter-redrob-reranker'

!python ml/train_reward_reranker_colab.py \
  --data-dir data/redrob-reranker \
  --base-model distilbert-base-uncased \
  --output-dir outputs/sifter-redrob-reranker \
  --hub-model-id "$MODEL_ID" \
  --epochs 1 \
  --batch-size 8 \
  --learning-rate 2e-5 \
  --precision fp32 \
  --push-to-hub

Optional: train an LLM-style DPO preference explainer from `dpo_train.jsonl`. This is heavier than the reward reranker.

In [ ]:
# Optional DPO step. Uncomment when you have enough GPU memory.
# DPO_MODEL_ID = f'{HF_USERNAME}/sifter-redrob-dpo-explainer'
# !python ml/train_dpo_explainer_colab.py --data-dir data/redrob-reranker --base-model Qwen/Qwen2.5-0.5B-Instruct --output-dir outputs/sifter-redrob-dpo-explainer --hub-model-id "$DPO_MODEL_ID" --epochs 1 --batch-size 2 --learning-rate 5e-6 --push-to-hub

After training, create a Hugging Face Space with SDK `Gradio`, upload `ml/hf_space`, and set `SIFTER_RERANKER_MODEL` to your model id.